In [1]:
# Importing all necessary libraries

import pandas as pd
import numpy as np
import sys
import os
sys.path.insert(0, os.path.abspath('..')) 
from src.data_related.data_selector import *
from src.features.preprocessing import *
from src.models.trainers import *
from src.evaluation.leaderboards import *

In [2]:
# Loading the dataframe
df = pd.read_csv('../datasets/processed/pkdx_min.csv', index_col='id')

# Reorganise the columns for easier reading and dropping names
tdf = df[['generation', 'type_1', 'type_2', 'rarity', 'stage', 'shape', 'color', 'height', 'weight', 'total_stats']].copy()


# Defining Gen10 test starters
new_gen_initials = pd.DataFrame({
    'name':['Browt', 'Pombon', 'Gecqua'],
    'generation': [10, 10, 10],
    'type_1': ['grass', 'fire', 'water'],
    'type_2': ['None', 'None', 'None'],
    'rarity': ['regular', 'regular', 'regular'],
    'stage': ['s1c3', 's1c3', 's1c3'],
    'shape': ['legs', 'quadruped', 'quadruped'],
    'color': ['yellow', 'red', 'blue'],
    'height': [0.3, 0.4, 0.3],
    'weight': [3.5, 6.7, 4.3]
})

In [3]:
data_ready = prepare_data(tdf, new_gen_initials)
metrics_ld = metrics_leaderboard(data_ready)
stats_ld = stats_prediction(data_ready)



In [27]:
def leaderboard_to_png(leaderboard_df, color_up, img_path='leaderboard.png'):
    """Raw matplotlib.table() - NO pandas conflicts, NO index"""
    import matplotlib.pyplot as plt
    from matplotlib.table import Table
    
    # Pure data
    data_matrix = leaderboard_df.to_numpy()
    headers = list(leaderboard_df.columns)
    
    nrows, ncols = data_matrix.shape
    fig, ax = plt.subplots(figsize=(ncols * 1.3, nrows * 0.45 + 0.5))
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    ax.axis('off')
    
    # Raw matplotlib Table - bulletproof
    tbl = Table(ax, bbox=[0, 0, 1, 1])
    
    # Add headers (row 0)
    for j, header in enumerate(headers):
        tbl.add_cell(0, j, 1/ncols, 1/(nrows+1), text=header,
                     loc='center', facecolor=color_up)
        tbl[(0, j)].get_text().set(weight='bold', color='white')
    
    # Add data rows
    for i in range(nrows):
        for j in range(ncols):
            tbl.add_cell(i+1, j, 1/ncols, 1/(nrows+1), 
                        text=str(data_matrix[i, j]), loc='center')
    
    ax.add_table(tbl)
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(10)
    tbl.scale(1, 1.1)
    
    plt.savefig(img_path, dpi=500, bbox_inches='tight', pad_inches=0, facecolor='white')
    plt.close()


leaderboard_to_png(stats_ld, color_up = "#0c2eb6", img_path='../plots/stats_ld.png')
leaderboard_to_png(metrics_ld.drop('Features Relevance', axis=1), color_up = "#910606", img_path='../plots/metrics_ld.png')
    